# Zadanie 2: RAG na własnym korpusie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/02-rag/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-02`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 3.

Części:
1. Korpus z metadanymi, chunking i Chroma z filtrem.
2. Wyszukiwanie: wektory, BM25, hybryda, reranking i porównanie na zestawie testowym.
3. Pipeline RAG i ocena odpowiedzi przez LLM.

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

In [16]:
import os

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-4o-mini", "classic": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

czat: proxy (copilot/gpt-5-mini, klasyczny: copilot/gpt-5-mini) | embeddingi: ollama (embeddinggemma)


## Część 1. Korpus, chunking, Chroma

1. Korpus: **co najmniej 10 dokumentów** (każdy min. 400 znaków) z wybranej dziedziny, np. regulamin studiów, instrukcje sprzętu, przepisy kulinarne, dokumentacja biblioteki. Można też użyć korpusu z zajęć (`data/`) i dopisać do niego **5 własnych dokumentów**. Każdy dokument ma co najmniej jedno pole metadanych, po którym ma sens filtrować (kategoria, rok, wersja, język).
2. Chunking `RecursiveCharacterTextSplitter` z wybranym `chunk_size` i `chunk_overlap`; wypisanie liczby chunków i jednego przykładowego chunku z metadanymi.
3. Kolekcja Chroma i **jedno zapytanie**, dla którego wynik z filtrem `where` różni się od wyniku bez filtra. Wypisanie obu list.

In [17]:
from pathlib import Path

type Doc = dict[str, str]

def load_docs(folder: str) -> list[Doc]:
    docs = []
    for path in sorted(Path(folder).glob("*.md")):
        title, meta, _, *body = path.read_text(encoding="utf-8").split("\n")
        meta = dict(kv.split(": ") for kv in meta.split(" | "))
        docs.append({"id": path.stem, "title": title.lstrip("# "), "dzial": meta["dział"],
                     "rok": int(meta["rok"]), "text": "\n".join(body).strip()})
    return docs

docs = load_docs("data")
print(f"dokumentów: {len(docs)}, znaków razem: {sum(len(d['text']) for d in docs)}")
for d in docs[:5]:
    print(f"  {d['id']:<24} {d['dzial']:<8} {d['rok']}  {len(d['text'])} znaków")

dokumentów: 20, znaków razem: 13401
  benefity-2026            HR       2026  492 znaków
  benefity                 HR       2025  731 znaków
  biuro                    Administracja 2025  680 znaków
  delegacje                Finanse  2026  784 znaków
  hasla-i-klucze           IT       2026  768 znaków


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

type Chunk = dict[str, str | dict[str, str]]  # id, meta, text

def make_chunks(
        docs: list[dict[str, str]], size: int = 300, overlap: int = 50
    ) -> list[Chunk]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = []
    for d in docs:
        for i, part in enumerate(splitter.split_text(d["text"])):
            chunks.append({
				"id": f"{d['id']}#{i}", 
                "text": f"{d['title']}\n{part}",
                "meta": {
                    "doc": d["id"],
                    "dzial": d["dzial"],
                    "rok": d["rok"]
                }
            })
    return chunks

chunks = make_chunks(docs, size=250, overlap=50)
print(len(chunks), "chunków")
print(chunks[70]["id"], chunks[70]["meta"])
print(chunks[70]["text"])

78 chunków
vpn#1 {'doc': 'vpn', 'dzial': 'IT', 'rok': 2026}
VPN i dostęp do sieci firmowej
Najczęstsze błędy:
- VPN-719: certyfikat urządzenia wygasł. Należy otworzyć portal samoobsługowy, wybrać „Odnów certyfikat” i ponownie zaimportować konfigurację. Certyfikat jest ważny 180 dni.


In [19]:
import chromadb

type Embedding = list[float]

__cache: dict[str, Embedding] = {}
def embed(texts: list[str]) -> list[Embedding]:
    """Embeddingi dla listy tekstów (z cache)."""
    todo = [t for t in texts if t not in __cache]
    if todo:
        for t, d in zip(todo, embed_client.embeddings.create(model=EMBED, input=todo).data):
            __cache[t] = d.embedding
    return [__cache[t] for t in texts]

def build_collection(chunks: list[Chunk], name: str = "firma") -> chromadb.Collection:
    db = chromadb.Client()
    if name in [c.name for c in db.list_collections()]:
        db.delete_collection(name)
    col = db.create_collection(name, metadata={"hnsw:space": "cosine"})
    col.add(
        ids=[c["id"] for c in chunks],                  # id chunku
        documents=[c["text"] for c in chunks],          # content
		metadatas=[c["meta"] for c in chunks],          # metadata chunku/pliku
        embeddings=embed([c["text"] for c in chunks])   # embedding contentu
    )
    return col

collection = build_collection(chunks)
print("w kolekcji:", collection.count())

w kolekcji: 78


In [20]:
type VecSearch = tuple[str, float, dict[str, str]]  # id chunku, distance, metadata chunku

def vector_search(
        query: str, k: int = 3, where: dict[str, str] | None = None
    ) -> list[VecSearch]:
    res = collection.query(query_embeddings=embed([query]), n_results=k, where=where)
    return list(zip(res["ids"][0], res["distances"][0], res["metadatas"][0]))

q = "Jaki pakiet prywatnej opieki zdrowotnej oferuje firma?"
for where in [None, {"rok": {"$gte": 2026}}]:
    print("where =", where)
    for cid, dist, meta in vector_search(q, where=where):
        print(f"   {dist:.3f}  {cid:<22} {meta['dzial']} {meta['rok']}")

where = None
   0.384  benefity#1             HR 2025
   0.452  lumen-sync-cennik#1    Produkt 2026
   0.501  benefity-2026#0        HR 2026
where = {'rok': {'$gte': 2026}}
   0.452  lumen-sync-cennik#1    Produkt 2026
   0.501  benefity-2026#0        HR 2026
   0.537  benefity-2026#1        HR 2026


In [21]:
for chunk in filter(lambda c: c["id"] in ["benefity#1", "benefity-2026#1"], chunks):
	print(chunk)

{'id': 'benefity-2026#1', 'text': 'Benefity pracownicze (wersja 2026)\nfinansowany w 50% przez firmę, a pozostałą część opłaca pracownik. Pakiety rozszerzone i wizyty stomatologiczne pozostają dodatkowo płatne. Dopłata do karty sportowej wynosi 70%, natomiast budżet szkoleniowy pozostaje na poziomie 3000 zł rocznie.', 'meta': {'doc': 'benefity-2026', 'dzial': 'HR', 'rok': 2026}}
{'id': 'benefity#1', 'text': 'Benefity pracownicze\nPrywatna opieka medyczna obejmuje pakiet podstawowy finansowany w całości przez firmę. Rozszerzenie pakietu o stomatologię i rehabilitację jest płatne przez pracownika.', 'meta': {'doc': 'benefity', 'dzial': 'HR', 'rok': 2025}}


## Część 2. Porównanie metod wyszukiwania

1. Zestaw testowy: **co najmniej 10 pytań** z oznaczonym dokumentem, w którym jest odpowiedź. Co najmniej 3 pytania to parafrazy (inne słowa niż w dokumencie), co najmniej 2 zawierają nazwę własną, liczbę albo kod.
2. Cztery metody: wektory, BM25, hybryda (RRF), hybryda + reranking cross-encoderem.
3. Tabela: `hit@1`, `hit@3`, `MRR` dla każdej metody.
4. Ta sama tabela (tylko wektory) dla **dwóch innych** ustawień `chunk_size`.
5. W komórce markdown (3–5 zdań): która metoda wypadła najlepiej, na którym pytaniu metody się różniły i dlaczego.

In [22]:
EVAL = [
    (
        "Jaki procent kosztu podstawowego pakietu medycznego pokrywa firma po zmianach w 2026 roku?",
        "benefity-2026",
        "Firma finansuje 50% podstawowego pakietu prywatnej opieki medycznej, a pozostałą część opłaca pracownik.",
    ),
    (
        "Jaki dodatkowy sprzęt mogę zamówić oprócz laptopa służbowego?",
        "sprzet",
        "Pracownik może zamówić dodatkowo monitor, klawiaturę, mysz oraz dodatkową pamięć RAM.",
    ),
    (
        "Ile dni przed szkoleniem trzeba wysłać wniosek w Kadrus?",
        "szkolenia-2026",
        "Wniosek należy przesłać co najmniej dziesięć dni przed rozpoczęciem szkolenia.",
    ),
    (
        "Gdzie znajduje się zamykana stojakownia dla rowerów?",
        "rowerem-do-biura",
        "Stojakownia znajduje się przy wejściu do biura od strony ulicy Jasnej.",
    ),
    (
        "Jak pracownik uzyskuje dostęp do pryszniców i stojakowni?",
        "rowerem-do-biura",
        "Dostęp do zamykanej stojakowni wydaje recepcja po okazaniu identyfikatora, a prysznice są dostępne w szatni.",
    ),
    (
        "Po jakim czasie od zatrudnienia można aplikować w rekrutacji wewnętrznej?",
        "rekrutacja-wewnetrzna",
        "Można aplikować po zakończeniu trzymiesięcznego okresu próbnego.",
    ),
    (
        "Czy obecny przełożony może zablokować przesłanie formularza do rekrutacji wewnętrznej?",
        "rekrutacja-wewnetrzna",
        "Nie. Zgoda obecnego przełożonego nie jest wymagana do przesłania formularza.",
    ),
    (
        "Czy będę miał gdzie zaparkować auto koło biura?",
        "biuro",
        "Tak. Dla pracowników dostępny jest parking podziemny liczący 40 miejsc parkingowych, pod warunkiem rezerwacji miejsca.",
    ),
    (
        "Czy muszę osobiście dostarczyć przełożonemu e-ZLA?",
        "urlopy",
        "Zwolnienie lekarskie trafia do firmy automatycznie z systemu ZUS. Wystarczy poinformować przełożonego o nieobecności.",
    ),
    (
        "Jak rozwiązać błąd VPN-305?",
        "vpn",
        "Trzeba zaktualizować WireGuard do wersji wskazanej w portalu.",
    ),
    (
        "Ile wynosi dieta za pełną dobę podróży krajowej?",
        "delegacje",
        "Dieta za dobę podróży krajowej wynosi 45 zł.",
    ),
    (
        "Co należy zrobić po zgubieniu klucza sprzętowego YubiKey?",
        "hasla-i-klucze",
        "Zgubienie klucza trzeba natychmiast zgłosić na helpdesk z priorytetem P2. Dział IT zablokuje klucz i wyda zapasowy w ciągu jednego dnia roboczego.",
    ),
    (
        "Ile kosztuje plan Start Lumen Sync przy płatności rocznej?",
        "lumen-sync-cennik",
        "Plan Start kosztuje 19 zł netto za użytkownika miesięcznie przy płatności rocznej.",
    ),
    (
        "Co oznacza błąd LS-429 w Lumen Sync?",
        "lumen-sync-kody-bledow",
        "Błąd LS-429 oznacza przekroczenie limitu zapytań na minutę.",
    ),
    (
        "Jaki jest limit zapytań API dla planu Zespół?",
        "lumen-sync-limity-api",
        "Plan Zespół ma limit 600 zapytań na minutę, a maksymalny rozmiar pliku wynosi 2 GB.",
    ),
    (
        "W jakim terminie należy zgłosić podejrzenie naruszenia ochrony danych?",
        "ochrona-danych",
        "Podejrzenie naruszenia trzeba zgłosić inspektorowi ochrony danych niezwłocznie, najpóźniej w ciągu 24 godzin od wykrycia.",
    ),
    (
        "O której godzinie nowa osoba powinna przyjść do biura pierwszego dnia?",
        "onboarding",
        "Pierwszego dnia nowa osoba powinna przyjść do biura na godzinę 9:00.",
    ),
    (
        "Ile dni pracy zdalnej w tygodniu można wykonywać według zasad z 2026 roku?",
        "praca-zdalna-2026",
        "Według zasad z 2026 roku pracownik może pracować zdalnie do trzech dni w tygodniu.",
    ),
    (
        "Ile dni urlopu na żądanie przysługuje pracownikowi w roku?",
        "urlopy",
        "Pracownikowi przysługują 4 dni urlopu na żądanie w roku.",
    ),
    (
        "Jak szybko helpdesk powinien zareagować na zgłoszenie P1?",
        "zglaszanie-awarii",
        "Na zgłoszenie P1 dyżurny inżynier powinien zareagować w ciągu 15 minut, przez całą dobę.",
    ),
    (
        "Jaki procent ceny karty sportowej dopłaca firma według zasad z 2025 roku?",
        "benefity",
        "Firma dopłaca 80% ceny karty sportowej, a pozostała część jest potrącana z wynagrodzenia.",
    ),
]

In [23]:
import re
from rank_bm25 import BM25Okapi

STOP = set("a i w z na do o od po się że to jest są nie co jak ile czy dla przez lub albo oraz też mogę".split())

def tokenize(text: str) -> list[str]:
    words = re.findall(r"\w+(?:-\d+)?", text.lower())
    return [w if any(ch.isdigit() for ch in w) else w[:6] for w in words if w not in STOP]

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
CHUNK_IDS = [c["id"] for c in chunks]

def bm25_search(query: str, k = 3) -> list[tuple[str, float]]:
    scores = bm25.get_scores(tokenize(query))
    best = np.argsort(-scores)[:k]
    return [(CHUNK_IDS[i], float(scores[i])) for i in best]

In [24]:
def rrf(rankings: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for pos, cid in enumerate(ranking, start=1):
            scores[cid] = scores.get(cid, 0) + 1 / (k + pos)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(query: str, k: int = 3, depth: int = 20) -> list[str]:
    vec = [cid for cid, _, _ in vector_search(query, k=depth, where=None)]
    kw = [cid for cid, _ in bm25_search(query, k=depth)]
    return rrf([vec, kw])[:k]

In [25]:
import os, warnings
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_VERBOSITY="error")
warnings.filterwarnings("ignore")
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("sdadas/polish-reranker-base-ranknet")

CHUNK_CONTENT = {c["id"]: c["text"] for c in chunks}

def rerank(query: str, candidates: list[str], k: int = 3) -> list[str]:
    scores = reranker.predict([(query, CHUNK_CONTENT[cid]) for cid in candidates])
    order = np.argsort(-scores)[:k]
    return [(candidates[i], float(scores[i])) for i in order]

In [26]:
from collections.abc import Callable
import numpy as np

def doc_rank(ranked_chunk_ids: list[str], gold_doc: str):
    """Pozycja pierwszego chunku z właściwego dokumentu (None, jeśli go nie ma)."""
    for pos, cid in enumerate(ranked_chunk_ids, start=1):
        if cid.partition("#")[0] == gold_doc:
            return pos
    return None

def evaluate(search_fn: Callable[[list[str]], list[str]], k: int = 3):
    ranks = [doc_rank(search_fn(q), gold) for q, gold, _ in EVAL]
    hit = np.mean([r is not None and r <= k for r in ranks])
    mrr = np.mean([1 / r if r else 0 for r in ranks])
    return hit, mrr

METHODS = {
    "wektory":           lambda q: [cid for cid, _, _ in vector_search(q, k=10, where=None)],
    "BM25":              lambda q: [cid for cid, _ in bm25_search(q, k=10)],
    "hybryda (RRF)":     lambda q: hybrid_search(q, k=10),
    "hybryda + rerank":  lambda q: [cid for cid, _ in rerank(q, hybrid_search(q, k=10), k=10)],
}
print(f"{'metoda':<20}{'hit@1':>7}{'hit@3':>7}{'MRR':>7}")
for name, fn in METHODS.items():
    h1, mrr = evaluate(fn, k=1)
    h3, _ = evaluate(fn, k=3)
    print(f"{name:<20}{h1:>7.2f}{h3:>7.2f}{mrr:>7.3f}")

metoda                hit@1  hit@3    MRR
wektory                0.90   0.95  0.940
BM25                   0.81   0.95  0.886
hybryda (RRF)          0.90   1.00  0.944
hybryda + rerank       0.95   1.00  0.976


In [27]:
print(f"{'chunk_size':>10}{'overlap':>9}{'chunków':>9}{'hit@1':>7}{'MRR':>7}")
for size, overlap in [(150, 0), (300, 50), (600, 100), (1200, 0)]:
    chunks_v = make_chunks(docs, size, overlap)
    collection = build_collection(chunks_v)
    h1, mrr = evaluate(METHODS["wektory"], k=1)
    print(f"{size:>10}{overlap:>9}{len(chunks_v):>9}{h1:>7.2f}{mrr:>7.3f}")
collection = build_collection(chunks)  # powrót do wcześniejszej konfiguracji

chunk_size  overlap  chunków  hit@1    MRR
       150        0      127   0.90  0.952
       300       50       70   0.90  0.940
       600      100       38   0.86  0.921
      1200        0       20   0.86  0.929


*Wnioski:* Najlepiej wypadła hybryda z rerankingiem, co pokazuje, że połączenie wyszukiwania wektorowego i słownikowego oraz dodatkowa ocena trafności poprawiają kolejność wyników; najsłabszy był BM25, szczególnie dla pytań będących parafrazami treści dokumentów. Najlepszy wynik samego wyszukiwania wektorowego uzyskano dla `chunk_size = 150` i `overlap = 0`, prawdopodobnie przez zwięzłość informacji i krótką zawartość plików; zbyt duże fragmenty (1200 znaków) natomiast pogorszyły wynik, prawdopodobnie przez mieszanie kilku informacji w jednym chunku.

## Część 3. RAG i ocena odpowiedzi

1. Funkcja `rag(question)`: wyszukiwanie (dowolna metoda z części 2) → prompt z kontekstem → odpowiedź z identyfikatorami źródeł.
2. Odpowiedzi na **co najmniej 5 pytań** z zestawu testowego oraz **1 pytanie, na które korpus nie zawiera odpowiedzi** (oczekiwane „Nie wiem”).
3. LLM jako sędzia: wierność (faithfulness) i poprawność każdej odpowiedzi, z jednozdaniowym uzasadnieniem, oraz średnie.
4. W komórce markdown (2–4 zdania): czy któraś ocena sędziego jest błędna; jeśli wszystkie są trafne, jaki przypadek mógłby go zmylić.

In [28]:
SYSTEM = """Odpowiadasz na pytania pracowników na podstawie fragmentów dokumentów firmy.
Korzystasz wyłącznie z podanego kontekstu. Jeśli kontekst nie zawiera odpowiedzi, piszesz: "Nie wiem".
Odpowiedź: 1-2 zdania po polsku, na końcu identyfikatory źródeł w nawiasach kwadratowych."""

def rag(question: str, k: int = 3) -> tuple[str, list[str]]:  # response + k citations
    hits = rerank(q, hybrid_search(question, k=k), k=k)
    texts = collection.get(ids=[cid for cid, _ in hits])
    context = "\n\n".join(f"[{i}] {t}" for i, t in zip(texts["ids"], texts["documents"]))
    msg = client.chat.completions.create(model=CHAT, messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Kontekst:\n{context}\n\nPytanie: {question}"},
    ]).choices[0].message.content
    return msg, [cid for cid, _ in hits]

In [29]:
import json

JUDGE = """Oceniasz odpowiedź systemu RAG. Zwróć wyłącznie JSON:
{"faithful": 0 albo 1, "correct": 0 albo 1, "reason": "jedno zdanie"}
faithful = 1, jeśli każde twierdzenie odpowiedzi wynika z kontekstu.
correct = 1, jeśli odpowiedź zawiera informację z odpowiedzi wzorcowej (inne sformułowanie jest w porządku)."""

def rag_return_context(question: str, k: int = 5) -> tuple[str, str]:  # response + concat content of k cited documents
    answer, ids = rag(question, k=k)
    return answer, "\n\n".join(collection.get(ids=ids)["documents"])

def judge(question: str, answer: str, context: str, reference: str) -> dict[str, int | str]:
    out = client.chat.completions.create(model=CHAT, response_format={"type": "json_object"}, messages=[
        {"role": "system", "content": JUDGE},
        {"role": "user", "content": f"Pytanie: {question}\nKontekst:\n{context}\n\nOdpowiedź wzorcowa: {reference}\nOdpowiedź systemu: {answer}"},
    ])
    return json.loads(out.choices[0].message.content)

In [ ]:
QnA = [
    (
        "Czy w okolicy biura znajduje się jakiś parking?",
        "Tak. Dla pracowników dostępny jest parking podziemny liczący 40 miejsc parkingowych, pod warunkiem rezerwacji miejsca.",
    ),
    (
        "Jaki dodatkowy sprzęt mogę zamówić oprócz laptopa służbowego?",
        "Pracownik może zamówić dodatkowo monitor, klawiaturę, mysz za łączną kwotę do 1500 PLN.",
    ),
    (
        "Jaką część kosztu podstawowego pakietu medycznego pokrywa firma?",
        "Firma pokrywa 50% kosztu podstawowego pakietu opieki zdrowotnej.",
    ),
    (
        "Czy w biurze mogę gdzieś zostawić rower?",
        "Tak. Dla pracowników dostępna jest zamknięta stojakownia przy wejściu od ulicy Jasnej.",
    ),
    ("Jakie modele telefonów służbowych są dostępne dla pracowników?", "Nie wiem."),
]


results = []
for i, (q, reference) in enumerate(QnA, start=1):
    answer, context = rag_return_context(q)
    verdict = judge(q, answer, context, reference)
    results.append(verdict)
    print(
        f"{i}. {q}\n"
        f"  Odpowiedź referencyjna: {reference}\n"
        f"  Odp. systemu: {answer}\n"
        f"  Werdykt: (F={verdict['faithful']} C={verdict['correct']}) {verdict['reason']}",
        end="\n\n"
    )
print(
    f"wierność: {np.mean([r['faithful'] for r in results]):.2f}   poprawność: {np.mean([r['correct'] for r in results]):.2f}"
)

1. Czy w okolicy biura znajduje się jakiś parking?
  Odpowiedź referencyjna: Tak. Dla pracowników dostępny jest parking podziemny liczący 40 miejsc parkingowych, pod warunkiem rezerwacji miejsca.
  Odp. systemu: Tak, biuro ma podziemny parking z 40 miejscami. Przydział miejsc na kolejny miesiąc odbywa się przez formularz administracji składany do 25. dnia miesiąca. [biuro#2]
  Werdykt: (F=1 C=1) Odpowiedź zgodnie z kontekstem wskazuje podziemny parking z 40 miejscami oraz procedurę przydziału miejsc, co odpowiada odpowiedzi wzorcowej.

2. Jaki dodatkowy sprzęt mogę zamówić oprócz laptopa służbowego?
  Odpowiedź referencyjna: Pracownik może zamówić dodatkowo monitor, klawiaturę, mysz za łączną kwotę do 1500 PLN.
  Odp. systemu: Możesz zamówić monitor, klawiaturę i mysz z katalogu sprzętu, raz na trzy lata, w limicie 1500 zł. Sprzęt pozostaje własnością firmy. [sprzet#1]
  Werdykt: (F=1 C=1) Odpowiedź wymienia monitor, klawiaturę i mysz, podaje limit 1500 zł oraz częstotliwość zamówienia

*Komentarz do ocen sędziego:* Sędzia przyznał pełną wierność odpowiedziom, ponieważ system nie dopowiadał informacji spoza dostarczonego kontekstu. Względnie wysoka jest również poprawność, czyli system poradził sobie dobraniem odpowiednich źródeł do odpowiedzi.  
  
Przyjrzyjmy się pytaniu numer 4.  
Tak wygląda treść chunku `rowerem-do-biura#1`: 

	Dojazdy rowerem i prysznice\nsą dwa prysznice oraz suszarka do odzieży. Ze względów bezpieczeństwa roweru nie wolno zostawiać na korytarzu ani przypinać do ogrodzenia. Usterki stojaków i wyposażenia należy zgłaszać administracji przez formularz Helpdesk.

Odpowiedź systemu faktycznie jest więc wierna, gdyż fragment ten nie odpowiada na pytanie.
Tej można doszukać się w chunku `rowerem-do-biura#0`:

	Dojazdy rowerem i prysznice\nPracownicy dojeżdżający do biura rowerem mogą korzystać z zamykanej stojakowni znajdującej się przy wejściu od strony ulicy Jasnej. Dostęp wydaje recepcja po okazaniu identyfikatora. W szatni dostępne są dwa prysznice oraz suszarka do odzieży. Ze

Najwyraźniej jednak embedding tego tekstu odbiega znacząco od embeddingu pytania przez co
chunk ten nie został wybrany jako istotny nawet wykorzystując reranking z wyszukiwaniem hybrydowym.

Zmylić sędziego można na przykład gdy w korpusie znajdują się dokumenty podające różne wersje tej samej informacji która zmieniała się w czasie - w zależności który chunk zostanie lepiej wyłapany i do której wersji będzie odnosić się odpowiedź referencyjna F i C mogą zostać ocenione nieprawidłowo. Na przykładzie dokumentacji Lumen Sync mogłoby to być pytanie o ilość dni pracy zdalnej, która różni się w zależności od wersji dokumentu.

In [40]:
rerank(q, hybrid_search("Czy w biurze mogę gdzieś zostawić rower?", k=5), k=5)

[('praca-zdalna-2026#2', 0.131898432970047),
 ('praca-zdalna-2026#1', 0.09655436128377914),
 ('rowerem-do-biura#1', 0.09251312911510468),
 ('praca-zdalna-2024#1', 0.07881852984428406),
 ('biuro#1', 0.03589734435081482)]